In [2]:
#Load Dataset
import pandas as pd

data = r"D:\Nandhini\covid-19_vaccine_tweets_with_sentiment.csv"
df = pd.read_csv(data, encoding="latin1")
print(df.head())
print(df.columns)
print(df.shape)

              5  label                                         tweet_text
0  1.360340e+18      1  4,000 a day dying from the so called Covid-19 ...
1  1.382900e+18      2  Pranam message for today manifested in Dhyan b...
2  1.375670e+18      2  Hyderabad-based ?@BharatBiotech? has sought fu...
3  1.381310e+18      1  Confirmation that Chinese #vaccines "dont hav...
4  1.362170e+18      3  Lab studies suggest #Pfizer, #Moderna vaccines...
Index(['5', 'label', 'tweet_text'], dtype='str')
(6000, 3)


In [3]:
#Missing Values
print(df.isnull().sum())

5             0
label         0
tweet_text    0
dtype: int64


In [4]:
#Duplicates
print(df.duplicated().sum())

0


In [5]:
#Convert to Lower case
df["tweet_text"] = df["tweet_text"].str.lower()
df.columns = ["tweet_id", "label", "tweet_text"]

In [7]:
#Remove URLs
import re

df["tweet_text"] = df["tweet_text"].apply(
    lambda x: re.sub(r'http\S+|www\S+', '', str(x))
)

In [8]:
#Remove Punctuations
import string

df["tweet_text"] = df["tweet_text"].apply(
    lambda x: x.translate(
        str.maketrans('', '', string.punctuation)
    )
)

In [9]:
#Remove Mentions
import re

df["tweet_text"] = df["tweet_text"].apply(
    lambda x: re.sub(r'@\w+', '', str(x))
)

print("Mentions Removed Successfully!")

Mentions Removed Successfully!


In [12]:
#Remove Stopwords
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

df["tweet_text"] = df["tweet_text"].apply(
    lambda x: " ".join(
        word for word in str(x).split()
        if word not in stop_words
    )
)

print("Stopwords Removed Successfully!")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


Stopwords Removed Successfully!


In [13]:
#Check Cleaned Tweets
print(df[["tweet_text"]].head())

                                          tweet_text
0  4000 day dying called covid19 vaccine dailyb...
1  pranam message today manifested dhyan meenapra...
2  hyderabadbased bharatbiotech sought funds gove...
3  confirmation chinese vaccines dont high prote...
4  lab studies suggest pfizer moderna vaccines pr...


In [14]:
#TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df["tweet_text"])

print("TF-IDF Shape :", X.shape)

TF-IDF Shape : (6000, 5000)


In [15]:
#Target Variable
y = df["label"]

print(y.head())

0    1
1    2
2    2
3    1
4    3
Name: label, dtype: int64


In [16]:
#Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Samples :", X_train.shape)
print("Testing Samples :", X_test.shape)

Training Samples : (4800, 5000)
Testing Samples : (1200, 5000)


In [17]:
#Model Training
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

print("Model Trained Successfully!")

Model Trained Successfully!


In [19]:
#Prediction
y_pred = lr.predict(X_test)

In [20]:
#Accuracy
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy :", accuracy)

Accuracy : 0.7383333333333333


In [23]:
#Classification Report
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.75      0.04      0.08        74
           2       0.75      0.90      0.82       759
           3       0.71      0.54      0.61       367

    accuracy                           0.74      1200
   macro avg       0.74      0.49      0.50      1200
weighted avg       0.74      0.74      0.71      1200



In [22]:
#Confusion Matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[  3  64   7]
 [  1 686  72]
 [  0 170 197]]


In [25]:
#Joblib
import joblib

joblib.dump(lr, "models/sentiment_model.pkl")
joblib.dump(tfidf, "models/tfidf_vectorizer.pkl")

print("Model Saved Successfully!")

FileNotFoundError: [Errno 2] No such file or directory: 'models/sentiment_model.pkl'